## 1. Imports and Configuration

In [22]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
import math
import os
import csv
import json
import re
import pandas as pd
import torch
from sam2.build_sam import build_sam2_video_predictor

print("✓ All libraries imported")

✓ All libraries imported


## Summary

All processing complete! Generated files:
- Field points: `outputs/field/field_points.json`, `outputs/field/map_points.json`
- Player tracking: `outputs/player_tracking/tracking_data.csv`, `outputs/player_tracking/tracking_summary.json`
- Ball tracking: `outputs/ball_sam2_track.csv` (if SAM2 available)
- Videos: `outputs/beach_volleyball_tracked.mp4`
- Trajectory maps: `outputs/trajectories.png`, `outputs/trajectory_player_*.png`

In [ ]:
# ============================================================
# VIDEO AND MODEL PATHS
# ============================================================
VIDEO_PATH = r"resources\\VideosAnalisis\\clip 3 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "resources\\beachvolleyballcourt.png"
MODEL_PATH = "weights\\yolo11n.pt"

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================
OUTPUT_BASE_DIR = "outputs"
OUTPUT_TRACKING_DIR = "outputs/player_tracking"
OUTPUT_FIELD_DIR = "outputs/field"
OUTPUT_BALL_DIR = "outputs/ball"

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_TRACKING_DIR, exist_ok=True)
os.makedirs(OUTPUT_FIELD_DIR, exist_ok=True)
os.makedirs(OUTPUT_BALL_DIR, exist_ok=True)

# ============================================================
# OUTPUT FILE NAMES
# ============================================================
# Field detection files
FIELD_POINTS_JSON = "field_points.json"
MAP_POINTS_JSON = "map_points.json"
FIELD_POINTS_PATH = f"{OUTPUT_FIELD_DIR}/{FIELD_POINTS_JSON}"
MAP_POINTS_PATH = f"{OUTPUT_FIELD_DIR}/{MAP_POINTS_JSON}"

# Player tracking files
TRACKING_DATA_CSV = "tracking_data.csv"
TRACKING_SUMMARY_JSON = "tracking_summary.json"
TRACKING_CSV_PATH = f"{OUTPUT_TRACKING_DIR}/{TRACKING_DATA_CSV}"
TRACKING_JSON_PATH = f"{OUTPUT_TRACKING_DIR}/{TRACKING_SUMMARY_JSON}"

# Ball tracking files
BALL_SAM2_TRACK_CSV = "ball_sam2_track.csv"
BALL_FIELD_TRAJECTORY_CSV = "ball_field_trajectory.csv"
OUT_CSV_BALL = f"{OUTPUT_BALL_DIR}/{BALL_SAM2_TRACK_CSV}"
BALL_FIELD_CSV = f"{OUTPUT_BALL_DIR}/{BALL_FIELD_TRAJECTORY_CSV}"

# Visualization files
TRAJECTORIES_PNG = "trajectories.png"
BALL_TRAJECTORY_FIELD_PNG = "ball_trajectory_field.png"
COMPLETE_TRACKING_VIDEO = "complete_tracking_video.mp4"
TRAJECTORIES_PATH = f"{OUTPUT_TRACKING_DIR}/{TRAJECTORIES_PNG}"
BALL_TRAJECTORY_PATH = f"{OUTPUT_BALL_DIR}/{BALL_TRAJECTORY_FIELD_PNG}"
OUTPUT_VIDEO_PATH = f"{OUTPUT_BASE_DIR}/{COMPLETE_TRACKING_VIDEO}"

# ============================================================
# FIELD DETECTION PARAMETERS
# ============================================================
AUTO_DETECT_FIELD = True  # True: automatic detection, False: manual selection
NUM_FRAMES_MEDIAN = 150    # Frames for average image generation
MARGIN_PERCENT = 0.10      # Expansion margin for detection zone

# ============================================================
# PLAYER TRACKING PARAMETERS
# ============================================================
EXPECTED_PLAYERS = 4
DETECTION_ZONE_EXPAND_X = 0.15
DETECTION_ZONE_EXPAND_Y = 0.25

TRACKER_MAX_AGE = 15
TRACKER_MIN_HITS = 3
TRACKER_IOU_THRESHOLD = 0.5
TRACKER_COLOR_WEIGHT = 0.2
TRACKER_POSITION_WEIGHT = 0.8
TRACKER_MIN_SIMILARITY = 0.3
TRACKER_MAX_MOVEMENT = 0.05

# ============================================================
# BALL TRACKING PARAMETERS (SAM2)
# ============================================================
SAM2_CKPT = os.path.join("checkpoints", "sam2.1_hiera_tiny.pt")
SAM2_CFG  = r"configs/sam2.1/sam2.1_hiera_t.yaml"
FRAMES_DIR = "./_sam2_frames"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROI_WINDOW_NAME   = "Selecciona la pelota (ROI)"
ROI_PREVIEW_MAX_W = 960
ROI_PREVIEW_MAX_H = 540
INIT_FRAME_GUESS  = 10
NONBLACK_SEARCH_LIMIT = 200

# Mezcla adaptativa (aplanado)
AIR_T0 = 0.20   # umbral bajo
AIR_T1 = 0.55   # umbral alto (>= => casi todo suelo)
SMOOTH_ALPHA_MAP = 0.35  # suavizado en minimapa (0..1)

# ============================================================
# BALL TRAJECTORY PROJECTION PARAMETERS
# ============================================================
CONTACT_DISTANCE_THRESHOLD = 100  # pixels - distance to consider ball-player contact
MIN_CONTACT_DURATION = 3          # frames - minimum frames to confirm contact

# ============================================================
# OUTPUT OPTIONS
# ============================================================
AUTO_OPEN_VIDEO = False
GENERATE_INDIVIDUAL_TRAJECTORIES = True

# ============================================================
# LOAD MODEL
# ============================================================
model = YOLO(MODEL_PATH)
print(f"✓ YOLO model loaded: {MODEL_PATH}")
print(f"✓ Video path: {VIDEO_PATH}")
print(f"✓ Map path: {MAPA_PATH}")
print(f"✓ Device: {DEVICE}")
print(f"✓ Output directories created")

✓ YOLO model loaded: weights\yolo11n.pt
✓ Video path: resources\\VideosAnalisis\\clip 3 ‐ Hecho con Clipchamp.mp4
✓ Map path: resources\beachvolleyballcourt.png
✓ Device: cpu
✓ Output directories created


## 2. Helper Functions

In [24]:
def get_points(event, x, y, flags, params):
    """Callback for mouse point selection."""
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]
    
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)
        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)

def expand_field_zone(field_points, expand_x=0.15, expand_y=0.20):
    """Expands the field zone for detection."""
    center = np.mean(field_points, axis=0)
    x_coords = field_points[:, 0]
    y_coords = field_points[:, 1]
    field_width = np.max(x_coords) - np.min(x_coords)
    field_height = np.max(y_coords) - np.min(y_coords)
    
    expanded_points = []
    for pt in field_points:
        direction = pt - center
        if abs(direction[0]) > 1e-6:
            expand_x_pixels = field_width * expand_x / 2
            direction[0] += np.sign(direction[0]) * expand_x_pixels
        if abs(direction[1]) > 1e-6:
            expand_y_pixels = field_height * expand_y / 2
            direction[1] += np.sign(direction[1]) * expand_y_pixels
        expanded_points.append(center + direction)
    
    return np.array(expanded_points, dtype=np.float32)

def point_in_polygon_with_margin(point, polygon, margin_percent=0.10):
    """Checks if a point is inside an expanded polygon."""
    center = np.mean(polygon, axis=0)
    expanded_polygon = []
    max_y = np.max(polygon[:, 1])
    
    for pt in polygon:
        if abs(pt[1] - max_y) < 5:
            direction = pt - center
            direction[1] = min(0, direction[1])
            expanded_pt = pt + direction * margin_percent
        else:
            direction = pt - center
            expanded_pt = pt + direction * margin_percent
        expanded_polygon.append(expanded_pt)
    
    expanded_polygon = np.array(expanded_polygon, dtype=np.int32)
    result = cv2.pointPolygonTest(expanded_polygon, point, False)
    return result >= 0

def calculate_iou(box1, box2):
    """Calculates IoU between two bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i < x1_i or y2_i < y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0

def get_player_color(track_id, all_ids, position_x=None, field_center_x=None):
    """Returns color for player based on team."""
    left_team_colors = [(255, 100, 0), (180, 0, 0)]
    right_team_colors = [(0, 100, 255), (0, 0, 180)]
    
    if position_x is not None and field_center_x is not None:
        sorted_ids = sorted(all_ids)
        if track_id in sorted_ids:
            team_player_idx = sorted_ids.index(track_id) % 2
            if position_x < field_center_x:
                return left_team_colors[team_player_idx]
            else:
                return right_team_colors[team_player_idx]
    
    return (200, 200, 200)

print("✓ Helper functions defined")

✓ Helper functions defined


## 2.1 Ball Tracking Helper Functions (SAM2)

In [25]:
def sanitize_folder_name(name):
    """Sanitize folder name by removing invalid characters."""
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r'[^\w\s\-.]', '_', name)
    name = re.sub(r'\s+', '_', name)
    return name.strip('_')

def extract_frames(video_path, frames_path):
    """Extract all frames from video to folder."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] Extrayendo {frame_count} frames...")
    for i in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            frame_count = i
            break
        cv2.imwrite(os.path.join(frames_path, f"{i:06d}.jpg"), frame)
        if i % 100 == 0:
            print(f"[INFO] Extraídos {i}/{frame_count} frames...")
    cap.release()
    print(f"[INFO] Extracción completada: {frame_count} frames guardados en {frames_path}")
    return fps, W, H, frame_count

def read_frame_from_video(video_path, frame_idx):
    """Read a single frame from video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

def is_black_frame(frame, thr_mean=8.0):
    """Check if frame is black."""
    if frame is None:
        return True
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(gray.mean()) < thr_mean

def pick_nonblack_frame_idx(video_path, start_idx, frame_count, search_limit=200):
    """Find first non-black frame starting from start_idx."""
    start_idx = max(0, min(int(start_idx), max(0, frame_count - 1)))
    max_idx = min(frame_count - 1, start_idx + int(search_limit))
    for idx in range(start_idx, max_idx + 1):
        fr = read_frame_from_video(video_path, idx)
        if fr is not None and not is_black_frame(fr):
            return idx, fr
    return start_idx, read_frame_from_video(video_path, start_idx)

def load_or_create_frame_jpg(frames_path, video_path, frame_idx):
    """Load frame from jpg or extract from video."""
    frame_file = os.path.join(frames_path, f"{int(frame_idx):06d}.jpg")
    if os.path.exists(frame_file):
        fr = cv2.imread(frame_file)
        if fr is not None:
            return fr
    fr = read_frame_from_video(video_path, frame_idx)
    if fr is None:
        return None
    cv2.imwrite(frame_file, fr)
    return fr

def resize_for_preview(frame, max_w=960, max_h=540):
    """Resize frame for preview window."""
    H, W = frame.shape[:2]
    scale = min(max_w / W, max_h / H, 1.0)
    new_w = int(round(W * scale))
    new_h = int(round(H * scale))
    preview = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA) if scale < 1.0 else frame.copy()
    sx = W / new_w
    sy = H / new_h
    return preview, sx, sy

def select_roi_small_window(frame_bgr):
    """Interactive ROI selection for ball."""
    preview, sx, sy = resize_for_preview(frame_bgr, ROI_PREVIEW_MAX_W, ROI_PREVIEW_MAX_H)
    cv2.namedWindow(ROI_WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(ROI_WINDOW_NAME, preview.shape[1], preview.shape[0])
    print("[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)")
    roi = cv2.selectROI(ROI_WINDOW_NAME, preview, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow(ROI_WINDOW_NAME)

    x, y, w, h = roi
    if w == 0 or h == 0:
        raise RuntimeError("ROI vacío. Vuelve a ejecutar y selecciona una caja válida.")

    x0 = float(x) * sx
    y0 = float(y) * sy
    x1 = float(x + w) * sx
    y1 = float(y + h) * sy
    return np.array([x0, y0, x1, y1], dtype=np.float32)

def mask_points(binmask: np.ndarray):
    """Extract centroid and ground point from mask."""
    ys, xs = np.where(binmask > 0)
    if len(xs) == 0:
        return (np.nan, np.nan, np.nan, np.nan, 0, 0.0)

    # centroide
    cx = float(xs.mean())
    cy = float(ys.mean())

    # "suelo" (punto inferior)
    y_max = int(ys.max())
    xs_bottom = xs[ys == y_max]
    xg = float(xs_bottom.mean()) if len(xs_bottom) else cx
    yg = float(y_max)

    # bbox aproximado para diametro
    x_min, x_max = int(xs.min()), int(xs.max())
    y_min, y_max2 = int(ys.min()), int(ys.max())
    bw = max(1, x_max - x_min + 1)
    bh = max(1, y_max2 - y_min + 1)
    diam = float(max(bw, bh))  # diámetro px aprox

    area = int(len(xs))
    return (cx, cy, xg, yg, area, diam)

def clamp01(x):
    """Clamp value to [0,1]."""
    return float(max(0.0, min(1.0, x)))

def mix2(a, b, w):
    """Linear interpolation between a and b."""
    return (1.0 - w) * a + w * b

def air_weight(cy, yg, diam):
    """
    Estimate how much ball is in air using vertical separation.
    Returns w in [0,1] (0 use centroid; 1 use ground)
    """
    if not np.isfinite(cy) or not np.isfinite(yg) or not np.isfinite(diam) or diam <= 1e-6:
        return 0.0
    score = (float(yg) - float(cy)) / float(diam)
    # map score [AIR_T0..AIR_T1] -> w [0..1]
    w = (score - AIR_T0) / (AIR_T1 - AIR_T0 + 1e-9)
    return clamp01(w)

print("✓ Ball tracking helper functions defined")

✓ Ball tracking helper functions defined


## 3. Automatic Field Detection

In [26]:
# ============================================================
# FIELD AND MAP SETUP
# ============================================================

# Load video and map
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"✓ Video loaded: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")

mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")

print(f"✓ Map loaded: {mapa.shape[1]}x{mapa.shape[0]}")

# ============================================================
# LOAD OR SELECT MAP POINTS
# ============================================================

if os.path.exists(MAP_POINTS_PATH):
    print(f"✓ Loading map points from {MAP_POINTS_PATH}")
    with open(MAP_POINTS_PATH, 'r') as f:
        map_data = json.load(f)
        puntos_mapa = np.array(map_data['map_points'], dtype=np.float32)
    print(f"✓ Map points loaded: {len(puntos_mapa)} points")
else:
    print("⚠️ Map points file not found. Please select 4 corners on the map.")
    puntos_mapa = []
    
    imgB = mapa.copy()
    cv2.namedWindow("Selecciona 4 esquinas en el mapa", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Selecciona 4 esquinas en el mapa", 800, 600)
    cv2.imshow("Selecciona 4 esquinas en el mapa", imgB)
    
    cv2.setMouseCallback(
        "Selecciona 4 esquinas en el mapa",
        get_points,
        {"points": puntos_mapa, "image": imgB,
         "wname": "Selecciona 4 esquinas en el mapa", "max_points": 4}
    )
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    puntos_mapa = np.array(puntos_mapa, dtype=np.float32)
    
    # Save map points for future use
    map_data_export = {'map_points': puntos_mapa.tolist()}
    with open(MAP_POINTS_PATH, 'w') as f:
        json.dump(map_data_export, f, indent=2)
    print(f"✓ Map points saved to {MAP_POINTS_PATH}")

# ============================================================
# DETECT OR SELECT FIELD CORNERS
# ============================================================

if AUTO_DETECT_FIELD:
    print("🔍 Automatic field detection...")
    
    # ============================================================
    # 1. GENERACIÓN DE LA IMAGEN MEDIA (REDUCCIÓN DE RUIDO)
    # ============================================================
    def generar_imagen_media(video_cap, num_frames):
        video_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        acc = None
        count = 0
        while count < num_frames:
            ret, frame = video_cap.read()
            if not ret:
                break
            frame_f = frame.astype(np.float32)
            acc = frame_f if acc is None else acc + frame_f
            count += 1
        if count == 0:
            raise ValueError("No se pudieron leer frames del vídeo.")
        avg = (acc / count).astype(np.uint8)
        video_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        return avg
    
    print(f"  Generating median image from {NUM_FRAMES_MEDIAN} frames...")
    avg = generar_imagen_media(video, NUM_FRAMES_MEDIAN)
    
    # ============================================================
    # 2. SEGMENTACIÓN DE LA ARENA EN ESPACIO HSV
    # ============================================================
    hsv = cv2.cvtColor(avg, cv2.COLOR_BGR2HSV)
    lower_sand = np.array([10, 25, 135])
    upper_sand = np.array([35, 140, 255])
    mask_sand = cv2.inRange(hsv, lower_sand, upper_sand)
    
    # ============================================================
    # 3. LIMPIEZA MORFOLÓGICA Y ROI ESPACIAL
    # ============================================================
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    mask_clean = cv2.morphologyEx(mask_sand, cv2.MORPH_CLOSE, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel)
    
    h_img, w_img = mask_clean.shape
    roi = np.zeros_like(mask_clean)
    margen_sup = 0.60
    margen_inf = 0.10
    margen_izq = 0.05
    margen_der = 0.05
    
    roi[int(h_img * margen_sup) : int(h_img * (1 - margen_inf)), 
        int(w_img * margen_izq) : int(w_img * (1 - margen_der))] = 255
    mask_roi = cv2.bitwise_and(mask_clean, roi)
    
    kernel_horizontal = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 15))
    mask_joined = cv2.morphologyEx(mask_roi, cv2.MORPH_CLOSE, kernel_horizontal)
    
    # ============================================================
    # 4. CÁLCULO DEL CENTRO DEL CAMPO
    # ============================================================
    ys, xs = np.where(mask_joined > 0)
    if len(xs) == 0:
        raise ValueError("No se pudo calcular el centro del campo")
    cx_field = int(xs.mean())
    
    kernel_small = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    mask_joined = cv2.morphologyEx(mask_joined, cv2.MORPH_OPEN, kernel_small)
    
    # ============================================================
    # 5. DETECCIÓN DEL CONTORNO Y CUADRILÁTERO
    # ============================================================
    contours, _ = cv2.findContours(mask_joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    main_cnt = max(contours, key=cv2.contourArea) if contours else None
    if main_cnt is None:
        raise ValueError("No se pudo detectar el campo")
    
    epsilon = 0.01 * cv2.arcLength(main_cnt, True)
    approx = cv2.approxPolyDP(main_cnt, epsilon, True)
    if len(approx) != 4:
        rect = cv2.minAreaRect(main_cnt)
        quad = cv2.boxPoints(rect).astype(int)
    else:
        quad = approx.reshape(4, 2)
    
    # ============================================================
    # 6. BORDES Y PREPARACIÓN DE HOUGH
    # ============================================================
    edges_source = cv2.GaussianBlur(mask_sand.copy(), (15, 11), 0)
    edges_raw = cv2.Canny(edges_source, 50, 150)
    mask_field = np.zeros_like(edges_raw)
    cv2.fillPoly(mask_field, [quad.astype(int)], 255)
    edges_in_field = cv2.bitwise_and(edges_raw, mask_field)
    
    # ============================================================
    # 7. FILTRADO DE LÍNEAS
    # ============================================================
    lines_h = cv2.HoughLinesP(edges_in_field, 1, np.pi/180, 75, minLineLength=100, maxLineGap=20)
    margin_h = np.deg2rad(7.5)
    lineas_horizontal = []
    if lines_h is not None:
        for x1, y1, x2, y2 in lines_h.reshape(-1,4):
            angle = math.atan2(y2 - y1, x2 - x1) % np.pi
            if abs(angle) < margin_h or abs(angle - np.pi) < margin_h:
                lineas_horizontal.append((x1, y1, x2, y2))
    
    lines_v = cv2.HoughLines(edges_in_field, 1, np.pi/180, threshold=60)
    margin_v = np.deg2rad(55)
    lineas_vertical = []
    
    if lines_v is not None:
        for l in lines_v:
            rho, theta = l[0]
            if abs(theta) < margin_v or abs(theta - np.pi) < margin_v:
                a, b = np.cos(theta), np.sin(theta)
                x0, y0 = a*rho, b*rho
                x1, y1 = int(x0 + 2000*(-b)), int(y0 + 2000*a)
                x2, y2 = int(x0 - 2000*(-b)), int(y0 - 2000*a)
                lineas_vertical.append((x1, y1, x2, y2))
    
    # ============================================================
    # 8. UNIFICACIÓN DE LÍNEAS
    # ============================================================
    def unificar_segmentos(segmentos, tol_rho=15, tol_theta=0.1):
        unificadas = []
        for seg in segmentos:
            x1, y1, x2, y2 = seg
            angle = math.atan2(y2 - y1, x2 - x1) % np.pi
            rho = (x1*y2 - x2*y1) / (np.hypot(x2-x1, y2-y1) + 1e-6)
            agregada = False
            for i, (urho, utheta, useg) in enumerate(unificadas):
                if abs(rho - urho) < tol_rho and abs(angle - utheta) < tol_theta:
                    new_seg = (int((x1+useg[0])/2), int((y1+useg[1])/2), 
                              int((x2+useg[2])/2), int((y2+useg[3])/2))
                    unificadas[i] = ((rho+urho)/2, (angle+utheta)/2, new_seg)
                    agregada = True
                    break
            if not agregada:
                unificadas.append((rho, angle, (x1, y1, x2, y2)))
        return [seg for _,_,seg in unificadas]
    
    lineas_unificadas_h = unificar_segmentos(lineas_horizontal, tol_rho=5, tol_theta=0.05)
    lineas_unificadas_v = unificar_segmentos(lineas_vertical, tol_rho=15, tol_theta=0.15)
    
    # ============================================================
    # 9. INTERSECCIONES DE LÍNEAS
    # ============================================================
    def obtener_interseccion(l1, l2):
        x1, y1, x2, y2 = l1
        x3, y3, x4, y4 = l2
        den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if abs(den) < 1e-6:
            return None
        px = ((x1*y2 - y1*x2)*(x3 - x4) - (x1 - x2)*(x3*y4 - y3*x4)) / den
        py = ((x1*y2 - y1*x2)*(y3 - y4) - (y1 - y2)*(x3*y4 - y3*x4)) / den
        return int(px), int(py)
    
    margen_central = w_img * 0.33
    v_filtradas = [l for l in lineas_unificadas_v
                   if ((l[0]+l[2])/2 < (w_img/2 - margen_central/2)) or
                      ((l[0]+l[2])/2 > (w_img/2 + margen_central/2))]
    
    intersecciones = []
    for lh in lineas_unificadas_h:
        for lv in v_filtradas:
            pt = obtener_interseccion(lh, lv)
            if pt:
                x, y = pt
                if -100 <= x <= w_img + 100 and -100 <= y <= h_img + 100:
                    intersecciones.append(pt)
    
    if len(intersecciones) < 4:
        print(f"⚠️ Only {len(intersecciones)} intersections found, falling back to manual selection")
        AUTO_DETECT_FIELD = False
    else:
        # ============================================================
        # 10. SELECCIÓN DE ESQUINAS POR CUADRANTES
        # ============================================================
        cy_field = int(h_img * 0.75)
        centro_ref = (cx_field, cy_field)
        
        candidatas_tl, candidatas_tr, candidatas_bl, candidatas_br = [], [], [], []
        
        for pt in intersecciones:
            px, py = pt
            if px < cx_field and py < cy_field:
                candidatas_tl.append(pt)
            elif px >= cx_field and py < cy_field:
                candidatas_tr.append(pt)
            elif px < cx_field and py >= cy_field:
                candidatas_bl.append(pt)
            elif px >= cx_field and py >= cy_field:
                candidatas_br.append(pt)
        
        def obtener_mas_cercano(lista, ref):
            return min(lista, key=lambda p: math.dist(p, ref)) if lista else None
        
        top_left     = obtener_mas_cercano(candidatas_tl, centro_ref)
        top_right    = obtener_mas_cercano(candidatas_tr, centro_ref)
        bottom_left  = obtener_mas_cercano(candidatas_bl, centro_ref)
        bottom_right = obtener_mas_cercano(candidatas_br, centro_ref)
        
        # Reflejo de seguridad (Fallback)
        if not top_left and top_right:
            top_left = (2*cx_field - top_right[0], top_right[1])
        if not top_right and top_left:
            top_right = (2*cx_field - top_left[0], top_left[1])
        if not bottom_left and bottom_right:
            bottom_left = (2*cx_field - bottom_right[0], bottom_right[1])
        if not bottom_right and bottom_left:
            bottom_right = (2*cx_field - bottom_left[0], bottom_left[1])
        
        esquinas = [top_left, top_right, bottom_right, bottom_left]
        
        if any(v is None for v in esquinas):
            print("⚠️ Could not determine all 4 corners, falling back to manual selection")
            AUTO_DETECT_FIELD = False
        else:
            puntos_campo = np.array(esquinas, dtype=np.float32)
            print(f"✓ Automatic detection successful: {len(puntos_campo)} corners found")

if not AUTO_DETECT_FIELD:
    print("👆 Manual field corner selection...")
    puntos_campo = []
    
    imgA = first_frame.copy()
    cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
    cv2.imshow("Selecciona 4 esquinas del campo", imgA)
    
    cv2.setMouseCallback(
        "Selecciona 4 esquinas del campo",
        get_points,
        {"points": puntos_campo, "image": imgA, 
         "wname": "Selecciona 4 esquinas del campo", "max_points": 4}
    )
    
    print("Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    puntos_campo = np.array(puntos_campo, dtype=np.float32)

assert len(puntos_campo) == 4 and len(puntos_mapa) == 4, "Error: Need 4 field and 4 map points"
print(f"✓ Field points: {len(puntos_campo)}, Map points: {len(puntos_mapa)}")

# ============================================================
# CALCULATE HOMOGRAPHY
# ============================================================

H, status = cv2.findHomography(puntos_campo, puntos_mapa, cv2.RANSAC)

if H is None:
    raise RuntimeError("No se pudo calcular la homografía")

print("✓ Homography calculated")

# ============================================================
# CALCULATE FIELD CENTER AND DETECTION ZONE
# ============================================================

field_center_x = int(np.mean(puntos_campo[:, 0]))
puntos_arena = expand_field_zone(puntos_campo, DETECTION_ZONE_EXPAND_X, DETECTION_ZONE_EXPAND_Y)

print(f"✓ Field center X = {field_center_x}")

# ============================================================
# SAVE FIELD POINTS
# ============================================================

FIELD_POINTS_PATH = f"{OUTPUT_FIELD_DIR}/field_points.json"

field_data = {
    'field_points': puntos_campo.tolist(),
    'detection_zone': puntos_arena.tolist(),
    'field_center_x': field_center_x
}

with open(FIELD_POINTS_PATH, 'w') as f:
    json.dump(field_data, f, indent=2)

print(f"✓ Field points saved to {FIELD_POINTS_PATH}")

✓ Video loaded: 1920x1080, 30.0 FPS, 442 frames
✓ Map loaded: 1536x1024
✓ Loading map points from outputs/field/map_points.json
✓ Map points loaded: 4 points
🔍 Automatic field detection...
  Generating median image from 150 frames...
✓ Automatic detection successful: 4 corners found
✓ Field points: 4, Map points: 4
✓ Homography calculated
✓ Field center X = 969
✓ Field points saved to outputs/field/field_points.json


## 4. PlayerTracker Class

In [27]:
class PlayerTracker:
    """Multi-player tracking with position and color features."""
    def __init__(self, max_age=30, min_hits=3, iou_threshold=0.3, 
                 color_weight=0.2, position_weight=0.8, max_players=4,
                 max_movement_percent=0.08, field_center_x=None):
        self.max_age = max_age
        self.min_hits = min_hits
        self.color_weight = color_weight
        self.position_weight = position_weight
        self.max_players = max_players
        self.max_movement = max_movement_percent
        self.field_center_x = field_center_x
        self.tracks = {}
        self.next_id = 1
        self.frame_count = 0
        self.diagonal = None
    
    def _init_frame(self, frame):
        if self.diagonal is None:
            h, w = frame.shape[:2]
            self.diagonal = np.sqrt(w**2 + h**2)
    
    def _extract_histogram(self, frame, bbox):
        x1, y1, x2, y2 = [max(0, int(v)) for v in bbox]
        x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
        if x2 <= x1 or y2 <= y1:
            return None
        
        roi = frame[y1:y2, x1:x2]
        h, w = roi.shape[:2]
        mx, my = int(w * 0.15), int(h * 0.1)
        if mx > 0 and my > 0 and h > my*2 and w > mx*2:
            roi = roi[my:h-my, mx:w-mx]
        if roi.size == 0:
            return None
        
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        hist_h = cv2.calcHist([hsv], [0], None, [50], [0, 180])
        hist_s = cv2.calcHist([hsv], [1], None, [60], [0, 256])
        cv2.normalize(hist_h, hist_h, 0, 1, cv2.NORM_MINMAX)
        cv2.normalize(hist_s, hist_s, 0, 1, cv2.NORM_MINMAX)
        return np.concatenate([hist_h.flatten(), hist_s.flatten()])
    
    def _compare_histograms(self, h1, h2):
        if h1 is None or h2 is None:
            return 0.5
        corr_h = cv2.compareHist(h1[:50].reshape(-1,1).astype(np.float32), 
                                  h2[:50].reshape(-1,1).astype(np.float32), cv2.HISTCMP_CORREL)
        corr_s = cv2.compareHist(h1[50:].reshape(-1,1).astype(np.float32), 
                                  h2[50:].reshape(-1,1).astype(np.float32), cv2.HISTCMP_CORREL)
        return (0.6 * corr_h + 0.4 * corr_s + 1) / 2
    
    def _predict_position(self, track):
        positions = track['positions']
        if len(positions) < 2:
            return positions[-1]
        
        recent = positions[-min(4, len(positions)):]
        velocities = [(recent[i][0] - recent[i-1][0], recent[i][1] - recent[i-1][1]) 
                      for i in range(1, len(recent))]
        weights = list(range(1, len(velocities) + 1))
        total_w = sum(weights)
        
        avg_v = (sum(v[0]*w for v,w in zip(velocities, weights)) / total_w,
                 sum(v[1]*w for v,w in zip(velocities, weights)) / total_w)
        
        frames_missed = max(1, track['age'])
        max_move = self.diagonal * self.max_movement * frames_missed
        
        return (positions[-1][0] + np.clip(avg_v[0] * frames_missed, -max_move, max_move),
                positions[-1][1] + np.clip(avg_v[1] * frames_missed, -max_move, max_move))
    
    def _calc_similarity(self, track, det_feat, pred_pos):
        det_pos = det_feat['position']
        last_pos = track['positions'][-1]
        frames_missed = max(1, track['age'] + 1)
        max_dist = self.diagonal * self.max_movement * frames_missed
        
        dist_pred = np.sqrt((det_pos[0] - pred_pos[0])**2 + (det_pos[1] - pred_pos[1])**2)
        dist_last = np.sqrt((det_pos[0] - last_pos[0])**2 + (det_pos[1] - last_pos[1])**2)
        
        absolute_max = min(300, max_dist * 3)
        if dist_last > absolute_max:
            return 0, 0, 0
        
        if dist_last > max_dist * 1.5:
            return 0, 0, 0
        
        pos_score = max(0, 1 - dist_pred / max_dist) * 0.7 + max(0, 1 - dist_last / max_dist) * 0.3
        if len(track['positions']) < 3:
            pos_score = max(0, 1 - dist_pred / max_dist) * 0.3 + max(0, 1 - dist_last / max_dist) * 0.7
        
        color_score = self._compare_histograms(track['histogram'], det_feat['histogram'])
        total = self.position_weight * pos_score + self.color_weight * color_score
        return total, pos_score, color_score
    
    def _create_track(self, det_feat):
        self.tracks[self.next_id] = {
            'id': self.next_id, 'bbox': det_feat['bbox'], 'position': det_feat['position'],
            'positions': [det_feat['position']], 'histogram': det_feat['histogram'],
            'age': 0, 'hits': 1, 'matched': True
        }
        self.next_id += 1
    
    def _update_track(self, track, det_feat):
        track['bbox'] = det_feat['bbox']
        track['position'] = det_feat['position']
        track['positions'].append(det_feat['position'])
        if track['histogram'] is not None and det_feat['histogram'] is not None:
            track['histogram'] = 0.2 * det_feat['histogram'] + 0.8 * track['histogram']
        elif det_feat['histogram'] is not None:
            track['histogram'] = det_feat['histogram']
        track['age'] = 0
        track['hits'] += 1
        track['matched'] = True
    
    def _count_players_per_side(self):
        if self.field_center_x is None:
            return None, None
        left_count = right_count = 0
        for t in self.tracks.values():
            if t['hits'] >= self.min_hits:
                if t['position'][0] < self.field_center_x:
                    left_count += 1
                else:
                    right_count += 1
        return left_count, right_count
    
    def _can_create_in_side(self, position_x):
        if self.field_center_x is None:
            return True
        left_count, right_count = self._count_players_per_side()
        if position_x < self.field_center_x:
            return left_count < 2
        else:
            return right_count < 2
    
    def _would_cross_field(self, track, new_position_x):
        if self.field_center_x is None:
            return False
        if track['hits'] < 1:
            return False
        
        tolerance = 100
        left_boundary = self.field_center_x - tolerance
        right_boundary = self.field_center_x + tolerance
        old_x = track['position'][0]
        
        old_in_left_zone = old_x < left_boundary
        old_in_right_zone = old_x >= right_boundary
        new_in_left_zone = new_position_x < left_boundary
        new_in_right_zone = new_position_x >= right_boundary
        
        crosses_left_to_right = old_in_left_zone and new_in_right_zone
        crosses_right_to_left = old_in_right_zone and new_in_left_zone
        return crosses_left_to_right or crosses_right_to_left
    
    def update(self, frame, detections):
        self.frame_count += 1
        self._init_frame(frame)
        
        for track in self.tracks.values():
            track['predicted_pos'] = self._predict_position(track)
            track['matched'] = False
        
        det_feats = [{'bbox': (d[0],d[1],d[2],d[3]), 'position': (d[4],d[5]),
                      'histogram': self._extract_histogram(frame, (d[0],d[1],d[2],d[3]))} 
                     for d in detections]
        
        if not self.tracks:
            if self.field_center_x and len(det_feats) >= self.max_players:
                det_feats_sorted = sorted(det_feats, key=lambda df: df['position'][0])
                left_dets = [df for df in det_feats_sorted if df['position'][0] < self.field_center_x][:2]
                right_dets = [df for df in det_feats_sorted if df['position'][0] >= self.field_center_x][:2]
                for df in left_dets + right_dets:
                    self._create_track(df)
            else:
                for df in det_feats[:self.max_players]:
                    self._create_track(df)
            return self._get_active()
        
        if not det_feats:
            self._age_tracks()
            return self._get_active()
        
        track_ids = list(self.tracks.keys())
        cost = np.full((len(track_ids), len(det_feats)), 1000.0)
        
        for i, tid in enumerate(track_ids):
            t = self.tracks[tid]
            for j, df in enumerate(det_feats):
                if t['hits'] >= 1 and self._would_cross_field(t, df['position'][0]):
                    cost[i, j] = 1000.0
                    continue
                sim, pos_s, _ = self._calc_similarity(t, df, t['predicted_pos'])
                if pos_s > 0.1:
                    cost[i, j] = 1 - sim
        
        row_idx, col_idx = linear_sum_assignment(cost)
        matched_t, matched_d = set(), set()
        
        for i, j in zip(row_idx, col_idx):
            if cost[i, j] < 0.5:
                t = self.tracks[track_ids[i]]
                df = det_feats[j]
                dist = np.sqrt((df['position'][0] - t['positions'][-1][0])**2 + 
                               (df['position'][1] - t['positions'][-1][1])**2)
                frames_missed = max(1, t['age'] + 1)
                max_allowed = self.diagonal * self.max_movement * frames_missed * 2
                
                if self._would_cross_field(t, df['position'][0]):
                    continue
                
                if dist <= min(250, max_allowed):
                    self._update_track(t, df)
                    matched_t.add(i)
                    matched_d.add(j)
        
        unmatched_d = [j for j in range(len(det_feats)) if j not in matched_d]
        unmatched_t = [i for i in range(len(track_ids)) if i not in matched_t]
        confirmed = len([t for t in self.tracks.values() if t['hits'] >= self.min_hits])
        
        for j in unmatched_d:
            det_x = det_feats[j]['position'][0]
            if confirmed < self.max_players and len(self.tracks) < self.max_players:
                if self._can_create_in_side(det_x):
                    self._create_track(det_feats[j])
                    confirmed += 1
            else:
                best_i, best_c = None, 0.6
                for i in unmatched_t:
                    if cost[i, j] < best_c:
                        t = self.tracks[track_ids[i]]
                        df = det_feats[j]
                        if self._would_cross_field(t, df['position'][0]):
                            continue
                        dist = np.sqrt((df['position'][0] - t['positions'][-1][0])**2 + 
                                       (df['position'][1] - t['positions'][-1][1])**2)
                        if dist < 200:
                            best_c, best_i = cost[i, j], i
                
                if best_i is not None:
                    self._update_track(self.tracks[track_ids[best_i]], det_feats[j])
                    unmatched_t.remove(best_i)
        
        self._age_tracks(det_feats)
        return self._get_active()
    
    def _age_tracks(self, det_feats=None):
        to_del = []
        for tid, t in self.tracks.items():
            if not t['matched']:
                t['age'] += 1
                base_max_age = self.max_age * 3 if t['hits'] >= self.min_hits else self.max_age
                max_age = base_max_age
                
                if self.field_center_x is not None and t['hits'] >= self.min_hits and det_feats:
                    track_x = t['position'][0]
                    tolerance = 100
                    track_in_left = track_x < (self.field_center_x - tolerance)
                    track_in_right = track_x >= (self.field_center_x + tolerance)
                    
                    if track_in_left:
                        opposite_dets = [df for df in det_feats if df['position'][0] >= (self.field_center_x + tolerance)]
                        if opposite_dets:
                            max_age = self.max_age
                    elif track_in_right:
                        opposite_dets = [df for df in det_feats if df['position'][0] < (self.field_center_x - tolerance)]
                        if opposite_dets:
                            max_age = self.max_age
                
                if t['age'] > max_age:
                    to_del.append(tid)
        for tid in to_del:
            del self.tracks[tid]
    
    def _get_active(self):
        return [(t['id'], *t['bbox'], *t['position']) 
                for t in self.tracks.values() if t['hits'] >= self.min_hits]

print("✓ PlayerTracker class defined")

✓ PlayerTracker class defined


## 5. Player Tracking

In [28]:
# Initialize tracker
tracker = PlayerTracker(
    max_age=TRACKER_MAX_AGE,
    min_hits=TRACKER_MIN_HITS,
    iou_threshold=TRACKER_IOU_THRESHOLD,
    color_weight=TRACKER_COLOR_WEIGHT,
    position_weight=TRACKER_POSITION_WEIGHT,
    max_players=EXPECTED_PLAYERS,
    max_movement_percent=TRACKER_MAX_MOVEMENT,
    field_center_x=field_center_x
)

# Process video frames
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
tracking_data = {}

print(f"\nProcessing {total_frames} frames...")
print(f"  • Max players: {EXPECTED_PLAYERS} (2 per side)")
print(f"  • Field center: X={field_center_x}")
print(f"  • Position weight: {TRACKER_POSITION_WEIGHT*100:.0f}%, Color weight: {TRACKER_COLOR_WEIGHT*100:.0f}%\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # YOLO detection
    results = model(frame, verbose=False, classes=[0])
    
    # Extract detections in arena
    yolo_detections = []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
            
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = y2
            
            # Check if in detection zone
            in_arena = cv2.pointPolygonTest(puntos_arena.astype(np.int32), (cx, cy), False) >= 0
            
            if in_arena:
                yolo_detections.append((x1, y1, x2, y2, cx, cy))
    
    # Track
    tracked_players = tracker.update(frame, yolo_detections)
    tracking_data[frame_idx] = tracked_players
    
    if frame_idx % 100 == 0:
        progress = (frame_idx / total_frames) * 100
        n_tracks = len(tracker.tracks)
        n_confirmed = len([t for t in tracker.tracks.values() if t['hits'] >= TRACKER_MIN_HITS])
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.0f}%) - {len(tracked_players)} players | Tracks: {n_tracks} (confirmed: {n_confirmed})")
    
    frame_idx += 1

video.release()

print(f"\n✓ Tracking completed")

# Collect statistics
all_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_ids.add(det[0])

final_ids = sorted(all_ids)
total_detections = sum(len(dets) for dets in tracking_data.values())

print(f"  Detected IDs: {final_ids} ({len(final_ids)} unique)")
print(f"  Total detections: {total_detections}")

# Check limit
if len(final_ids) <= EXPECTED_PLAYERS:
    print(f"  ✅ Player limit of {EXPECTED_PLAYERS} respected")
else:
    print(f"  ⚠️ More IDs detected than expected ({len(final_ids)} vs {EXPECTED_PLAYERS})")

# Frames per player
id_frame_counts = defaultdict(int)
for dets in tracking_data.values():
    for det in dets:
        id_frame_counts[det[0]] += 1

print(f"\n  Frames per player:")
for track_id in sorted(id_frame_counts.keys()):
    frames = id_frame_counts[track_id]
    percentage = (frames / total_frames) * 100
    status = "✓" if percentage > 50 else "⚠️"
    print(f"    {status} ID {track_id}: {frames} frames ({percentage:.0f}%)")


Processing 442 frames...
  • Max players: 4 (2 per side)
  • Field center: X=969
  • Position weight: 80%, Color weight: 20%

  Frame 0/442 (0%) - 0 players | Tracks: 4 (confirmed: 0)
  Frame 100/442 (23%) - 4 players | Tracks: 4 (confirmed: 4)
  Frame 200/442 (45%) - 4 players | Tracks: 4 (confirmed: 4)
  Frame 300/442 (68%) - 4 players | Tracks: 4 (confirmed: 4)
  Frame 400/442 (90%) - 4 players | Tracks: 4 (confirmed: 4)

✓ Tracking completed
  Detected IDs: [1, 2, 3, 4] (4 unique)
  Total detections: 1760
  ✅ Player limit of 4 respected

  Frames per player:
    ✓ ID 1: 440 frames (100%)
    ✓ ID 2: 440 frames (100%)
    ✓ ID 3: 440 frames (100%)
    ✓ ID 4: 440 frames (100%)


## 6. Export Results (CSV and JSON)

In [29]:
# Export to CSV
tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'field_x': float(field_x),
            'field_y': float(field_y),
            'timestamp_sec': frame_idx / fps
        })

csv_filename = TRACKING_CSV_PATH
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'field_x', 'field_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tracking_list)

print(f"✓ CSV exported: {csv_filename}")

# Export to JSON
detections_per_id = {}
for tid in sorted(final_ids):
    detections_per_id[str(tid)] = id_frame_counts[tid]

summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height
    },
    'homography_matrix': H.tolist(),
    'player_ids': sorted(final_ids),
    'num_players': len(final_ids),
    'total_detections': {
        'total': len(tracking_list),
        'per_player': detections_per_id
    },
    'tracker_config': {
        'max_age': TRACKER_MAX_AGE,
        'min_hits': TRACKER_MIN_HITS,
        'min_similarity': TRACKER_MIN_SIMILARITY,
        'color_weight': TRACKER_COLOR_WEIGHT,
        'position_weight': TRACKER_POSITION_WEIGHT
    }
}

json_filename = TRACKING_JSON_PATH
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

print(f"✓ JSON exported: {json_filename}")

✓ CSV exported: outputs/player_tracking/tracking_data.csv
✓ JSON exported: outputs/player_tracking/tracking_summary.json


## 7. Generate Trajectory Maps

In [30]:
# Reconstruct trajectories
mapa_traj = cv2.imread(MAPA_PATH)
full_trajectories = defaultdict(list)

for frame_idx in sorted(tracking_data.keys()):
    for det in tracking_data[frame_idx]:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        full_trajectories[track_id].append((int(field_x), int(field_y), cx))

# Draw combined trajectories
for track_id in sorted(full_trajectories.keys()):
    trajectory = full_trajectories[track_id]
    
    if len(trajectory) < 2:
        continue
    
    # Lines
    for i in range(1, len(trajectory)):
        field_x1, field_y1, cx1 = trajectory[i-1]
        field_x2, field_y2, cx2 = trajectory[i]
        
        avg_cx = (cx1 + cx2) / 2
        color = get_player_color(track_id, final_ids, avg_cx, field_center_x)
        cv2.line(mapa_traj, (field_x1, field_y1), (field_x2, field_y2), color, 3)
    
    # Start point
    start_x, start_y, start_cx = trajectory[0]
    start_color = get_player_color(track_id, final_ids, start_cx, field_center_x)
    cv2.circle(mapa_traj, (start_x, start_y), 12, (255, 255, 255), -1)
    cv2.circle(mapa_traj, (start_x, start_y), 8, start_color, -1)
    cv2.putText(mapa_traj, f"{track_id}", (start_x - 10, start_y - 15),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # End point
    end_x, end_y, end_cx = trajectory[-1]
    end_color = get_player_color(track_id, final_ids, end_cx, field_center_x)
    cv2.rectangle(mapa_traj, (end_x-8, end_y-8), (end_x+8, end_y+8), end_color, -1)
    cv2.rectangle(mapa_traj, (end_x-10, end_y-10), (end_x+10, end_y+10), (255, 255, 255), 2)

# Legend
cv2.putText(mapa_traj, "LEFT TEAM", (20, 30),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 100, 0), 2)
cv2.circle(mapa_traj, (350, 25), 8, (255, 100, 0), -1)
cv2.circle(mapa_traj, (390, 25), 8, (180, 0, 0), -1)

cv2.putText(mapa_traj, "RIGHT TEAM", (20, 65),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 100, 255), 2)
cv2.circle(mapa_traj, (330, 60), 8, (0, 100, 255), -1)
cv2.circle(mapa_traj, (370, 60), 8, (0, 0, 180), -1)

# Save combined trajectories
traj_filename = f"{OUTPUT_TRACKING_DIR}/trajectories.png"
cv2.imwrite(traj_filename, mapa_traj)
print(f"✓ Combined trajectories saved: {traj_filename}")

# Generate individual trajectory images
if GENERATE_INDIVIDUAL_TRAJECTORIES:
    for track_id in sorted(full_trajectories.keys()):
        trajectory = full_trajectories[track_id]
        
        if len(trajectory) < 2:
            continue
        
        mapa_individual = cv2.imread(MAPA_PATH)
        
        avg_cx = np.mean([t[2] for t in trajectory])
        color = get_player_color(track_id, final_ids, avg_cx, field_center_x)
        
        # Draw trajectory
        for i in range(1, len(trajectory)):
            field_x1, field_y1, _ = trajectory[i-1]
            field_x2, field_y2, _ = trajectory[i]
            cv2.line(mapa_individual, (field_x1, field_y1), (field_x2, field_y2), color, 3)
        
        # Start
        start_x, start_y, _ = trajectory[0]
        cv2.circle(mapa_individual, (start_x, start_y), 12, (255, 255, 255), -1)
        cv2.circle(mapa_individual, (start_x, start_y), 8, color, -1)
        cv2.putText(mapa_individual, "START", (start_x - 25, start_y - 18),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # End
        end_x, end_y, _ = trajectory[-1]
        cv2.rectangle(mapa_individual, (end_x-8, end_y-8), (end_x+8, end_y+8), color, -1)
        cv2.rectangle(mapa_individual, (end_x-10, end_y-10), (end_x+10, end_y+10), (255, 255, 255), 2)
        cv2.putText(mapa_individual, "END", (end_x - 12, end_y - 18),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # Title
        team = "LEFT" if avg_cx < field_center_x else "RIGHT"
        cv2.putText(mapa_individual, f"PLAYER {track_id} - {team} TEAM", (20, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        
        individual_filename = f"{OUTPUT_TRACKING_DIR}/trajectory_player_{track_id}.png"
        cv2.imwrite(individual_filename, mapa_individual)
        print(f"  ✓ {individual_filename}")

✓ Combined trajectories saved: outputs/player_tracking/trajectories.png
  ✓ outputs/player_tracking/trajectory_player_1.png
  ✓ outputs/player_tracking/trajectory_player_2.png
  ✓ outputs/player_tracking/trajectory_player_3.png
  ✓ outputs/player_tracking/trajectory_player_4.png


## 8. Ball Tracking with SAM2

In [31]:
# Check if SAM2 is available
if not os.path.exists(SAM2_CKPT):
    print(f"⚠️ SAM2 checkpoint not found: {SAM2_CKPT}")
    print("   Download it from: https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt")
    print("   Or skip ball tracking")
else:
    print("✓ SAM2 checkpoint found")
    print("🏐 Starting ball tracking...")
    
    # Create frames directory
    video_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    video_stem_clean = sanitize_folder_name(video_stem)
    frames_path = os.path.join(FRAMES_DIR, video_stem_clean)
    os.makedirs(frames_path, exist_ok=True)
    
    # Extract frames if not exist
    existing_jpgs = [f for f in os.listdir(frames_path) if f.lower().endswith(".jpg")]
    if len(existing_jpgs) == 0:
        fps_ball, W_ball, H_ball, frame_count_ball = extract_frames(VIDEO_PATH, frames_path)
    else:
        cap0 = cv2.VideoCapture(VIDEO_PATH)
        if not cap0.isOpened():
            raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")
        fps_ball = cap0.get(cv2.CAP_PROP_FPS)
        W_ball = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
        H_ball = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_count_ball = int(cap0.get(cv2.CAP_PROP_FRAME_COUNT))
        cap0.release()
        print(f"✓ Using existing frames from {frames_path}")
    
    # Frame inicial no negro
    init_frame_idx, _ = pick_nonblack_frame_idx(
        VIDEO_PATH, start_idx=INIT_FRAME_GUESS, frame_count=frame_count_ball, search_limit=NONBLACK_SEARCH_LIMIT
    )
    init_frame = load_or_create_frame_jpg(frames_path, VIDEO_PATH, init_frame_idx)
    if init_frame is None:
        raise RuntimeError("No pude cargar frame inicial para ROI.")
    print(f"[INFO] Usando frame inicial: {init_frame_idx:06d}.jpg")
    
    # ROI selection
    box = select_roi_small_window(init_frame)
    print(f"[INFO] ROI seleccionado: {box}")
    
    # SAM2 predictor
    print(f"[INFO] Cargando SAM2 en {DEVICE} ...")
    predictor = build_sam2_video_predictor(SAM2_CFG, SAM2_CKPT, device=DEVICE)
    
    print(f"[INFO] Inicializando estado de SAM2 con carpeta: {frames_path}")
    with torch.inference_mode():
        try:
            state = predictor.init_state(video_path=frames_path)
        except TypeError:
            state = predictor.init_state(frames_path)
    
    OBJ_ID = 1
    with torch.inference_mode():
        try:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), obj_id=OBJ_ID, box=box)
        except TypeError:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), object_id=OBJ_ID, box=box)
    
    # Propagate
    print("[INFO] Propagando máscara en el vídeo...")
    rows = []
    with torch.inference_mode():
        for f_idx, obj_ids, masks in predictor.propagate_in_video(state):
            f_idx = int(f_idx)
            obj_ids_list = [int(o) for o in obj_ids]
            if OBJ_ID not in obj_ids_list:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": 0, "diam": np.nan})
                continue
            
            j = obj_ids_list.index(OBJ_ID)
            mask = masks[j]
            
            mask_np = mask.squeeze().detach().float().cpu().numpy()
            binmask = (mask_np > 0).astype(np.uint8)
            
            cx, cy, xg, yg, area, diam = mask_points(binmask)
            if area < 5:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": area, "diam": diam})
                continue
            
            rows.append({"frame": f_idx, "x": cx, "y": cy, "xg": xg, "yg": yg, "area": area, "diam": diam})
            
            if f_idx % 50 == 0:
                print(f"[INFO] Procesado frame {f_idx}/{frame_count_ball}")
    
    df = pd.DataFrame(rows).sort_values("frame")
    df.to_csv(OUT_CSV_BALL, index=False)
    print(f"[OK] CSV guardado: {OUT_CSV_BALL} (incluye xg,yg,diam)")
    print("✓ Ball tracking completed!")

✓ SAM2 checkpoint found
🏐 Starting ball tracking...
✓ Using existing frames from ./_sam2_frames\clip_3___Hecho_con_Clipchamp
[INFO] Usando frame inicial: 000010.jpg
[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)
[INFO] ROI seleccionado: [        558         554         588         578]
[INFO] Cargando SAM2 en cpu ...
[INFO] Inicializando estado de SAM2 con carpeta: ./_sam2_frames\clip_3___Hecho_con_Clipchamp


frame loading (JPEG): 100%|██████████| 448/448 [00:40<00:00, 10.95it/s]


[INFO] Propagando máscara en el vídeo...


propagate in video:   9%|▉         | 41/438 [04:33<52:29,  7.93s/it]

[INFO] Procesado frame 50/442


...

[INFO] Procesado frame 350/442


propagate in video:  89%|████████▉ | 391/438 [40:41<04:51,  6.20s/it]

[INFO] Procesado frame 400/442


propagate in video: 100%|██████████| 438/438 [45:34<00:00,  6.24s/it]

[OK] CSV guardado: outputs/ball/ball_sam2_track.csv (incluye xg,yg,diam)
✓ Ball tracking completed!


## 9. Ball Trajectory Projection to Field

Project ball trajectory to field coordinates using player contacts as reference points.

In [34]:
# ============================================================
# BALL TRAJECTORY PROJECTION TO FIELD COORDINATES
# ============================================================

print("🏐 Projecting ball trajectory to field coordinates...")

# Check if ball tracking CSV exists
if not os.path.exists(OUT_CSV_BALL):
    print(f"⚠️ Ball tracking CSV not found: {OUT_CSV_BALL}")
    print("   Run Section 9 (Ball Tracking) first")
else:
    # Load ball tracking data
    df_ball = pd.read_csv(OUT_CSV_BALL)
    print(f"✓ Loaded ball data: {len(df_ball)} frames")
    
    # Load player tracking data
    csv_players = TRACKING_CSV_PATH
    if not os.path.exists(csv_players):
        print(f"⚠️ Player tracking CSV not found: {csv_players}")
        print("   Run Section 7 (Export Results) first")
    else:
        df_players = pd.read_csv(csv_players)
        print(f"✓ Loaded player data: {len(df_players)} detections")
        
        # ============================================================
        # DETECT BALL-PLAYER CONTACTS
        # ============================================================
        print("\n📍 Detecting ball-player contacts...")
        
        contacts = []
        
        for idx, ball_row in df_ball.iterrows():
            frame = int(ball_row['frame'])
            ball_x = ball_row['x']
            ball_y = ball_row['y']
            
            # Skip if ball position is NaN
            if pd.isna(ball_x) or pd.isna(ball_y):
                continue
            
            # Get players in this frame
            players_in_frame = df_players[df_players['frame'] == frame]
            
            # Check distance to each player
            min_distance = float('inf')
            closest_player = None
            closest_player_field_pos = None
            
            for _, player_row in players_in_frame.iterrows():
                player_x = player_row['center_x']
                player_y = player_row['center_y']
                
                # Calculate distance
                distance = np.sqrt((ball_x - player_x)**2 + (ball_y - player_y)**2)
                
                if distance < min_distance:
                    min_distance = distance
                    closest_player = int(player_row['track_id'])
                    closest_player_field_pos = (player_row['field_x'], player_row['field_y'])
            
            # Register contact if within threshold
            if min_distance < CONTACT_DISTANCE_THRESHOLD and closest_player is not None:
                contacts.append({
                    'frame': frame,
                    'ball_x': ball_x,
                    'ball_y': ball_y,
                    'player_id': closest_player,
                    'distance': min_distance,
                    'field_x': closest_player_field_pos[0],
                    'field_y': closest_player_field_pos[1]
                })
        
        print(f"✓ Detected {len(contacts)} ball-player proximity events")
        
        # ============================================================
        # GROUP CONTACTS INTO CONTACT SEGMENTS
        # ============================================================
        print("\n🔗 Grouping contacts into segments...")
        
        contact_segments = []
        if contacts:
            current_segment = [contacts[0]]
            
            for i in range(1, len(contacts)):
                frame_diff = contacts[i]['frame'] - contacts[i-1]['frame']
                
                # If frames are consecutive or very close, continue segment
                if frame_diff <= MIN_CONTACT_DURATION:
                    current_segment.append(contacts[i])
                else:
                    # End current segment and start new one
                    if len(current_segment) >= MIN_CONTACT_DURATION:
                        contact_segments.append(current_segment)
                    current_segment = [contacts[i]]
            
            # Add last segment
            if len(current_segment) >= MIN_CONTACT_DURATION:
                contact_segments.append(current_segment)
        
        print(f"✓ Found {len(contact_segments)} contact segments")
        
        # ============================================================
        # PROJECT BALL TO FIELD COORDINATES
        # ============================================================
        print("\n🗺️ Projecting ball trajectory to field...")
        
        ball_field_trajectory = []
        
        for idx, ball_row in df_ball.iterrows():
            frame = int(ball_row['frame'])
            ball_x = ball_row['x']
            ball_y = ball_row['y']
            
            # Skip if ball position is NaN
            if pd.isna(ball_x) or pd.isna(ball_y):
                ball_field_trajectory.append({
                    'frame': frame,
                    'ball_x_video': np.nan,
                    'ball_y_video': np.nan,
                    'ball_x_field': np.nan,
                    'ball_y_field': np.nan,
                    'projection_method': 'missing'
                })
                continue
            
            # Check if this frame is part of a contact segment
            in_contact = False
            contact_field_pos = None
            
            for segment in contact_segments:
                segment_frames = [c['frame'] for c in segment]
                if frame in segment_frames:
                    in_contact = True
                    # Get average field position during this contact
                    contact_field_x = np.mean([c['field_x'] for c in segment])
                    contact_field_y = np.mean([c['field_y'] for c in segment])
                    contact_field_pos = (contact_field_x, contact_field_y)
                    break
            
            if in_contact and contact_field_pos:
                # Use player's field position
                ball_field_trajectory.append({
                    'frame': frame,
                    'ball_x_video': ball_x,
                    'ball_y_video': ball_y,
                    'ball_x_field': contact_field_pos[0],
                    'ball_y_field': contact_field_pos[1],
                    'projection_method': 'contact'
                })
            else:
                # Use homography to project ball position
                point_video = np.array([[[ball_x, ball_y]]], dtype=np.float32)
                point_field = cv2.perspectiveTransform(point_video, H)
                field_x, field_y = point_field[0][0]
                
                ball_field_trajectory.append({
                    'frame': frame,
                    'ball_x_video': ball_x,
                    'ball_y_video': ball_y,
                    'ball_x_field': float(field_x),
                    'ball_y_field': float(field_y),
                    'projection_method': 'homography'
                })
        
        # ============================================================
        # SAVE BALL FIELD TRAJECTORY
        # ============================================================
        df_ball_field = pd.DataFrame(ball_field_trajectory)
        ball_field_csv = BALL_FIELD_CSV
        df_ball_field.to_csv(ball_field_csv, index=False)
        print(f"✓ Ball field trajectory saved: {ball_field_csv}")
        
        # Statistics
        n_contact = len(df_ball_field[df_ball_field['projection_method'] == 'contact'])
        n_homography = len(df_ball_field[df_ball_field['projection_method'] == 'homography'])
        n_missing = len(df_ball_field[df_ball_field['projection_method'] == 'missing'])
        
        print(f"\n📊 Trajectory statistics:")
        print(f"  • Contact-based points: {n_contact} ({n_contact/len(df_ball_field)*100:.1f}%)")
        print(f"  • Homography-based points: {n_homography} ({n_homography/len(df_ball_field)*100:.1f}%)")
        print(f"  • Missing points: {n_missing} ({n_missing/len(df_ball_field)*100:.1f}%)")
        print(f"  • Contact segments: {len(contact_segments)}")
        
        # ============================================================
        # VISUALIZE BALL TRAJECTORY ON FIELD MAP
        # ============================================================
        print("\n🎨 Generating ball trajectory visualization...")
        
        mapa_ball = cv2.imread(MAPA_PATH)
        
        # Draw ball trajectory
        valid_points = df_ball_field[df_ball_field['projection_method'] != 'missing']
        
        for i in range(1, len(valid_points)):
            row1 = valid_points.iloc[i-1]
            row2 = valid_points.iloc[i]
            
            x1, y1 = int(row1['ball_x_field']), int(row1['ball_y_field'])
            x2, y2 = int(row2['ball_x_field']), int(row2['ball_y_field'])
            
            # Color based on projection method
            if row2['projection_method'] == 'contact':
                color = (0, 255, 0)  # Green for contact points
                thickness = 3
            else:
                color = (0, 165, 255)  # Orange for homography points
                thickness = 2
            
            cv2.line(mapa_ball, (x1, y1), (x2, y2), color, thickness)
        
        # Mark contact segments with circles
        for segment in contact_segments:
            # Get middle frame of segment
            mid_idx = len(segment) // 2
            mid_contact = segment[mid_idx]
            cx = int(mid_contact['field_x'])
            cy = int(mid_contact['field_y'])
            
            cv2.circle(mapa_ball, (cx, cy), 15, (0, 255, 0), -1)
            cv2.circle(mapa_ball, (cx, cy), 18, (255, 255, 255), 2)
            cv2.putText(mapa_ball, f"P{mid_contact['player_id']}", 
                       (cx - 15, cy - 25),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Legend
        cv2.putText(mapa_ball, "BALL TRAJECTORY", (20, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        cv2.rectangle(mapa_ball, (20, 50), (50, 70), (0, 255, 0), -1)
        cv2.putText(mapa_ball, "Contact with player", (60, 68),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.rectangle(mapa_ball, (20, 80), (50, 100), (0, 165, 255), -1)
        cv2.putText(mapa_ball, "Homography projection", (60, 98),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Save visualization
        ball_traj_img = BALL_TRAJECTORY_PATH
        cv2.imwrite(ball_traj_img, mapa_ball)
        print(f"✓ Ball trajectory visualization saved: {ball_traj_img}")
        
        print("\n✅ Ball trajectory projection completed!")

🏐 Projecting ball trajectory to field coordinates...
✓ Loaded ball data: 438 frames
✓ Loaded player data: 1760 detections

📍 Detecting ball-player contacts...
✓ Detected 146 ball-player proximity events

🔗 Grouping contacts into segments...
✓ Found 5 contact segments

🗺️ Projecting ball trajectory to field...
✓ Ball field trajectory saved: outputs/ball/ball_field_trajectory.csv

📊 Trajectory statistics:
  • Contact-based points: 141 (32.2%)
  • Homography-based points: 297 (67.8%)
  • Missing points: 0 (0.0%)
  • Contact segments: 5

🎨 Generating ball trajectory visualization...
✓ Ball trajectory visualization saved: outputs/ball/ball_trajectory_field.png

✅ Ball trajectory projection completed!


## 10. Generate Complete Tracking Video

Generate final video with player tracking, ball tracking, and minimap with ball trajectory.

In [35]:
# ============================================================
# GENERATE COMPLETE TRACKING VIDEO WITH BALL
# ============================================================

print("🎬 Generating complete tracking video...")

# Check if ball trajectory exists
BALL_ENABLED = os.path.exists(BALL_FIELD_CSV)
if BALL_ENABLED:
    df_ball_video = pd.read_csv(BALL_FIELD_CSV)
    print(f"✓ Ball trajectory loaded: {len(df_ball_video)} frames")
    
    # Extract contact frames (where ball touches player)
    contact_frames = set(df_ball_video[df_ball_video['projection_method'] == 'contact']['frame'].values)
    print(f"✓ Contact frames detected: {len(contact_frames)} frames with player touch")
else:
    df_ball_video = None
    contact_frames = set()
    print("⚠️ Ball trajectory not found - video will only show player tracking")

# Load video and map for output generation
video = cv2.VideoCapture(VIDEO_PATH)
output_filename = OUTPUT_VIDEO_PATH
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Setup output dimensions
map_display_width = width // 2
map_display_height = int(mapa.shape[0] * (map_display_width / mapa.shape[1]))
output_height = height + map_display_height
output_width = width

out = cv2.VideoWriter(output_filename, fourcc, fps, (output_width, output_height))

# Calculate scaling factors for map display
scale_x = map_display_width / mapa.shape[1]
scale_y = map_display_height / mapa.shape[0]

frame_idx = 0
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
trajectories = defaultdict(list)
ball_trajectory_map = []

print(f"\nGenerating video frames...")

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # ============================================================
    # TOP: VIDEO WITH TRACKING
    # ============================================================
    tracking_frame = frame.copy()
    cv2.polylines(tracking_frame, [puntos_arena.astype(np.int32)], True, (0, 255, 255), 2)
    cv2.polylines(tracking_frame, [puntos_campo.astype(np.int32)], True, (0, 255, 0), 2)
    
    # Draw players
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            color = get_player_color(track_id, final_ids, cx, field_center_x)
            
            cv2.rectangle(tracking_frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(tracking_frame, (cx, cy), 5, color, -1)
            cv2.putText(tracking_frame, f"ID {track_id}", (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Store trajectory for map
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa[0][0]
            trajectories[track_id].append((int(field_x), int(field_y), cx))
    
    # Draw ball on video
    if BALL_ENABLED:
        ball_data_frame = df_ball_video[df_ball_video['frame'] == frame_idx]
        if len(ball_data_frame) > 0:
            ball_row = ball_data_frame.iloc[0]
            ball_x = ball_row['ball_x_video']
            ball_y = ball_row['ball_y_video']
            
            if not pd.isna(ball_x) and not pd.isna(ball_y):
                bx, by = int(ball_x), int(ball_y)
                
                # Different colors based on contact status
                if frame_idx in contact_frames:
                    ball_color = (0, 255, 0)  # Green when touching player
                    radius = 15
                    thickness = 3
                    cv2.circle(tracking_frame, (bx, by), radius + 5, (255, 255, 255), 2)
                else:
                    ball_color = (0, 165, 255)  # Orange when in air
                    radius = 10
                    thickness = -1
                
                cv2.circle(tracking_frame, (bx, by), radius, ball_color, thickness)
                
                # Add contact indicator
                if frame_idx in contact_frames:
                    cv2.putText(tracking_frame, "CONTACT", (bx + 20, by - 20),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Info text
    num_players = len(tracking_data.get(frame_idx, []))
    contact_indicator = " | BALL CONTACT!" if frame_idx in contact_frames else ""
    info_text = f"Frame: {frame_idx}/{total_frames} | Players: {num_players}{contact_indicator}"
    cv2.putText(tracking_frame, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # ============================================================
    # BOTTOM: MINIMAP WITH PLAYERS AND BALL TRAJECTORY
    # ============================================================
    mapa_resized = cv2.resize(mapa, (map_display_width, map_display_height))
    mapa_display = mapa_resized.copy()
    
    # Draw ball trajectory on map (all history up to current frame)
    if BALL_ENABLED:
        ball_history = df_ball_video[df_ball_video['frame'] <= frame_idx]
        ball_history = ball_history[ball_history['projection_method'] != 'missing']
        
        for i in range(1, len(ball_history)):
            row1 = ball_history.iloc[i-1]
            row2 = ball_history.iloc[i]
            
            x1 = int(row1['ball_x_field'] * scale_x)
            y1 = int(row1['ball_y_field'] * scale_y)
            x2 = int(row2['ball_x_field'] * scale_x)
            y2 = int(row2['ball_y_field'] * scale_y)
            
            # Color based on projection method
            if row2['projection_method'] == 'contact':
                line_color = (0, 255, 0)  # Green for contact
                line_thickness = 3
            else:
                line_color = (0, 165, 255)  # Orange for flight
                line_thickness = 2
            
            cv2.line(mapa_display, (x1, y1), (x2, y2), line_color, line_thickness)
        
        # Draw current ball position on map
        ball_current = df_ball_video[df_ball_video['frame'] == frame_idx]
        if len(ball_current) > 0:
            ball_row = ball_current.iloc[0]
            if ball_row['projection_method'] != 'missing':
                ball_map_x = int(ball_row['ball_x_field'] * scale_x)
                ball_map_y = int(ball_row['ball_y_field'] * scale_y)
                
                if frame_idx in contact_frames:
                    cv2.circle(mapa_display, (ball_map_x, ball_map_y), 12, (0, 255, 0), -1)
                    cv2.circle(mapa_display, (ball_map_x, ball_map_y), 15, (255, 255, 255), 2)
                else:
                    cv2.circle(mapa_display, (ball_map_x, ball_map_y), 8, (0, 165, 255), -1)
                    cv2.circle(mapa_display, (ball_map_x, ball_map_y), 10, (255, 255, 255), 2)
    
    # Draw current player positions on map
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            color = get_player_color(track_id, final_ids, cx, field_center_x)
            
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa_pos = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa_pos[0][0]
            
            # Scale position for display
            current_pos = (int(field_x * scale_x), int(field_y * scale_y))
            
            cv2.circle(mapa_display, current_pos, 8, color, -1)
            cv2.circle(mapa_display, current_pos, 10, (255, 255, 255), 2)
            cv2.putText(mapa_display, f"{track_id}", 
                       (current_pos[0] + 12, current_pos[1] - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Legend on minimap
    legend_y = 20
    cv2.putText(mapa_display, "MINIMAP", (10, legend_y),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    if BALL_ENABLED:
        legend_y += 25
        cv2.circle(mapa_display, (20, legend_y), 6, (0, 165, 255), -1)
        cv2.putText(mapa_display, "Ball (flight)", (35, legend_y + 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        legend_y += 20
        cv2.circle(mapa_display, (20, legend_y), 6, (0, 255, 0), -1)
        cv2.putText(mapa_display, "Ball (contact)", (35, legend_y + 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # ============================================================
    # COMBINE FRAMES
    # ============================================================
    combined_frame = np.zeros((output_height, output_width, 3), dtype=np.uint8)
    combined_frame[0:height, 0:width] = tracking_frame
    
    map_x_offset = (output_width - map_display_width) // 2
    combined_frame[height:height+map_display_height, 
                   map_x_offset:map_x_offset+map_display_width] = mapa_display
    
    out.write(combined_frame)
    
    if frame_idx % 100 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Progress: {progress:.0f}% (Frame {frame_idx}/{total_frames})")
    
    frame_idx += 1

video.release()
out.release()

print(f"\n✅ Complete tracking video saved: {output_filename}")
print(f"   • Total frames: {frame_idx}")
print(f"   • Resolution: {output_width}x{output_height}")
print(f"   • Players tracked: {len(final_ids)}")
if BALL_ENABLED:
    print(f"   • Ball tracking: Enabled ({len(contact_frames)} contact frames)")
else:
    print(f"   • Ball tracking: Disabled (run Section 8-9 to enable)")

if AUTO_OPEN_VIDEO:
    import subprocess
    subprocess.Popen(['start', output_filename], shell=True)

🎬 Generating complete tracking video...
✓ Ball trajectory loaded: 438 frames
✓ Contact frames detected: 141 frames with player touch

Generating video frames...
  Progress: 0% (Frame 0/442)
  Progress: 23% (Frame 100/442)
  Progress: 45% (Frame 200/442)
  Progress: 68% (Frame 300/442)
  Progress: 90% (Frame 400/442)

✅ Complete tracking video saved: outputs/player_tracking/complete_tracking_video.mp4
   • Total frames: 442
   • Resolution: 1920x1720
   • Players tracked: 4
   • Ball tracking: Enabled (141 contact frames)
